# Evaluating an AI Agent with DeepEval

An agent isn't judged by prose alone — it **plans, picks tools, and acts**. DeepEval has metrics
for each part of that loop:

| Metric | What it checks | Needs |
| --- | --- | --- |
| `ToolCorrectnessMetric` | Did it call the *right* tools? (deterministic name match) | `tools_called`, `expected_tools` |
| `ArgumentCorrectnessMetric` | Were the *arguments* passed to each tool correct? (LLM judge) | `input`, `tools_called` |
| `GEval` (custom) | Did it actually accomplish what the user asked, end to end? | `input`, `actual_output` |

We'll build a tiny developer-assistant agent with three tools, let Groq decide which to call, and
score the result.

## 1. Install packages

We only need three libraries: `deepeval` itself, `google-genai` (so DeepEval's judge can talk to
Gemini), and `groq` (the model we're evaluating). No OpenAI key anywhere in this notebook.

In [1]:
%pip install -q deepeval google-genai groq


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 2. Add your API keys

Two roles, kept separate the whole way through this course:

- **Groq** — the *model under test*. It free-tier serves `llama-3.3-70b-versatile`. Get a key at
  https://console.groq.com/keys
- **Gemini** — the *judge*. Every DeepEval metric needs an LLM to grade with, and we use Gemini so
  you never need an OpenAI key. Get a free key at https://aistudio.google.com/apikey

Keys are entered with `getpass` so they never get printed or saved into the notebook file.

In [2]:
import os, getpass

os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")  # skip DeepEval's anonymous telemetry

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your GEMINI_API_KEY: ")
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]  # some libs read this name instead

print("Keys set.")

Keys set.


## 3. Define a couple of tools

Plain Python functions with canned, deterministic outputs — the lesson here is *evaluating which
tool the agent chooses*, not building real tool backends.

In [3]:
import random

def search_docs(query: str = "") -> str:
    return f"Docs result for '{query}': found a section with a definition and a short example."

def run_code(code: str = "") -> str:
    if "2**10" in code:
        return "Executed in a sandbox. stdout: 1024"
    return "Executed in a sandbox. stdout: (result)"

def escalate_to_human(reason: str = "") -> str:
    return f"Escalated to a human engineer. Ticket AI-{random.randint(1000, 9999)}. Reason: {reason}"

TOOLS = {"search_docs": search_docs, "run_code": run_code, "escalate_to_human": escalate_to_human}

TOOL_SCHEMAS = [
    {"type": "function", "function": {"name": "search_docs",
        "description": "Search internal documentation for a concept or policy.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "run_code",
        "description": "Execute a Python snippet in a sandbox and return its stdout.",
        "parameters": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}}},
    {"type": "function", "function": {"name": "escalate_to_human",
        "description": "Escalate the issue to a human engineer.",
        "parameters": {"type": "object", "properties": {"reason": {"type": "string"}}, "required": ["reason"]}}},
]

## 4. Groq tool-calling: let the model choose

The model proposes a tool + arguments; **we** execute it and feed the result back for the final
answer. We capture every call as a `ToolCall` — that's the object DeepEval's agent metrics read.

In [4]:
import json
import re
import uuid
from types import SimpleNamespace

from groq import Groq, BadRequestError
from deepeval.test_case import ToolCall

groq_client = Groq()
AGENT_SYSTEM = (
    "You are a GenAI developer assistant. Use the available tools to fulfil the user's request. "
    "Call the tool(s) that actually accomplish the task -- do not guess something you can look up "
    "or run."
)


def _parse_failed_generation(text: str):
    # Groq's Llama tool-calling models sometimes emit a malformed pseudo tool-call as raw text
    # (e.g. `<function=search_docs{"query": "..."}</function>`) instead of a structured tool_call,
    # and Groq's own server then fails to parse it -- a 400 'tool_use_failed' BadRequestError.
    # The intended call is still recoverable from the error's `failed_generation` field.
    m = re.match(r"^<function=(\w+)(\{.*\})\s*</function>\s*$", (text or "").strip())
    if not m:
        return None
    name, args_json = m.group(1), m.group(2)
    return SimpleNamespace(id=f"call_{uuid.uuid4().hex[:8]}", type="function",
                            function=SimpleNamespace(name=name, arguments=args_json))


def _first_completion(model: str, messages: list):
    try:
        return groq_client.chat.completions.create(
            model=model, messages=messages, tools=TOOL_SCHEMAS, tool_choice="auto", temperature=0,
        ).choices[0].message
    except BadRequestError as e:
        body = getattr(e, "body", None) or {}
        error = body.get("error", {})
        if error.get("code") != "tool_use_failed":
            raise
        parsed_call = _parse_failed_generation(error.get("failed_generation", ""))
        if parsed_call is None:
            raise
        return SimpleNamespace(content="", tool_calls=[parsed_call])


def run_agent(query: str, model: str = "llama-3.3-70b-versatile") -> dict:
    messages = [{"role": "system", "content": AGENT_SYSTEM}, {"role": "user", "content": query}]

    first = _first_completion(model, messages)

    calls = first.tool_calls or []
    tools_called = []
    messages.append({"role": "assistant", "content": first.content or "", "tool_calls": [
        {"id": c.id, "type": "function", "function": {"name": c.function.name, "arguments": c.function.arguments}}
        for c in calls
    ]})

    for c in calls:
        name = c.function.name
        args = json.loads(c.function.arguments or "{}")
        output = TOOLS[name](**args)
        tools_called.append(ToolCall(name=name, input_parameters=args, output=output))
        messages.append({"role": "tool", "tool_call_id": c.id, "content": output})

    if calls:
        final = groq_client.chat.completions.create(model=model, messages=messages, temperature=0).choices[0].message.content
    else:
        final = first.content or "(no tool called)"

    return {"actual_output": final.strip(), "tools_called": tools_called,
            "tool_names": [tc.name for tc in tools_called]}


## 5. Try the agent on a few requests

One doc lookup, one code execution, and one that should trigger **two** tools in sequence.

In [5]:
CASES = [
    {"input": "What do the docs say about the difference between fine-tuning and prompting?",
     "expected_tools": ["search_docs"]},
    {"input": "Run this and tell me the output: print(2**10)",
     "expected_tools": ["run_code"]},
    {"input": "My fine-tuning job keeps failing silently. Check the docs for known issues, and if "
              "that doesn't explain it, get a human to look at it.",
     "expected_tools": ["search_docs", "escalate_to_human"]},
]

agent_rows = []
for c in CASES:
    result = run_agent(c["input"])
    agent_rows.append({**c, **result})
    print("Q:", c["input"])
    print("Tools called:", result["tool_names"], " (expected:", c["expected_tools"], ")")
    print("Answer:", result["actual_output"])
    print("-" * 80)

Q: What do the docs say about the difference between fine-tuning and prompting?
Tools called: ['search_docs']  (expected: ['search_docs'] )
Answer: Fine-tuning and prompting are two different approaches to adapting a pre-trained language model to a specific task or dataset.

Fine-tuning involves updating the model's weights to fit the new task or dataset. This is typically done by adding a new layer on top of the pre-trained model and training the entire model on the new data. Fine-tuning can be computationally expensive and requires a significant amount of labeled data.

Prompting, on the other hand, involves providing the model with a prompt or input that guides it to generate a specific output. This approach does not require updating the model's weights and can be done with a small amount of data or even no data at all. Prompting can be used for a variety of tasks, such as text generation, question answering, and sentiment analysis.

In summary, fine-tuning is a more intensive proce

Q: Run this and tell me the output: print(2**10)
Tools called: ['run_code']  (expected: ['run_code'] )
Answer: The output is: 1024
--------------------------------------------------------------------------------


Q: My fine-tuning job keeps failing silently. Check the docs for known issues, and if that doesn't explain it, get a human to look at it.
Tools called: ['search_docs', 'escalate_to_human']  (expected: ['search_docs', 'escalate_to_human'] )
Answer: I've checked the documentation for known issues related to fine-tuning jobs failing silently, but unfortunately, it didn't provide a solution. I've escalated the issue to a human engineer for further assistance. Your ticket number is AI-4633. Someone will be in touch with you soon to help resolve the issue.
--------------------------------------------------------------------------------


## 6. Build `LLMTestCase`s with `tools_called` + `expected_tools`

In [6]:
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input=r["input"],
        actual_output=r["actual_output"],
        tools_called=r["tools_called"],
        expected_tools=[ToolCall(name=n) for n in r["expected_tools"]],
    )
    for r in agent_rows
]

## The judge: `GeminiModel`

Every DeepEval metric defaults to OpenAI if you don't tell it otherwise — even a metric that
scores deterministically will still try to build an OpenAI client unless you pass `model=`. We
build **one Gemini judge** here and pass it to every metric in this notebook.

In [7]:
from deepeval.models import GeminiModel

judge = GeminiModel(model="gemini-2.5-flash", api_key=os.environ["GEMINI_API_KEY"], temperature=0)
print("Judge ready:", judge.get_model_name())

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Judge ready: gemini-2.5-flash (Gemini)


## 8. Define the agent metrics

`ToolCorrectnessMetric` scores deterministically (it's just comparing names), but its constructor
still builds a fallback client if you don't pass `model=` — so pass the judge everywhere, even here.

In [8]:
from deepeval.metrics import ArgumentCorrectnessMetric, GEval, ToolCorrectnessMetric
from deepeval.test_case import SingleTurnParams

metrics = [
    ToolCorrectnessMetric(model=judge),
    ArgumentCorrectnessMetric(model=judge),
    GEval(
        name="Task Completion",
        criteria="Given the user request (input), did the final answer (actual_output) fully and "
                 "correctly accomplish what the user asked?",
        evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
        model=judge,
    ),
]

## 9. Run the evaluation

In [9]:
from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig, DisplayConfig, ErrorConfig

results = evaluate(
    test_cases=test_cases,
    metrics=metrics,
    async_config=AsyncConfig(max_concurrent=2),
    display_config=DisplayConfig(print_results=False),
    error_config=ErrorConfig(ignore_errors=True, skip_on_missing_params=True),
)

✨ You're running DeepEval's latest Tool Correctness Metric! (using None, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Argument Correctness Metric! (using gemini-2.5-flash (Gemini), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Task Completion [GEval] Metric! (using gemini-2.5-flash (Gemini), strict=False,
async_mode=True)...

Output()

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.

...

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x108cfca40> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x108cfca40> is already entered

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x108cfca40> is already entered

⚠ WARNING: No hyperparameters logged.
» ]8;id=751633;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.97s | token cost: 7.59e-05 USD)
» Test Results (3 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## Read the results

`evaluate()` returns an `EvaluationResult` with one `TestResult` per test case. Each `TestResult`
carries a `metrics_data` list — one entry per metric, with a `score`, a `reason` (the judge's
explanation), and whether it `success`-fully passed its threshold.

If a metric's judge call itself fails (a rate limit, an auth error, a transient API error), the
metric has **no score** — `score` is `None`, and the real explanation lives in `.error` instead of
`.reason`. Always guard for that instead of formatting `.score` directly, or you'll get a
`TypeError: unsupported format string passed to NoneType.__format__` in place of the real error.

If you see `⚠ WARNING: All metrics errored across every test case`, every metric hit the same
failure — almost always an invalid/expired `GEMINI_API_KEY`, a Gemini free-tier rate limit (5
metrics x N test cases fires a burst of judge calls), or a package version mismatch in whichever
environment/kernel you're running (re-check `pip show deepeval google-genai`). Print
`results.test_results[0].metrics_data[0].error` to see the real exception immediately.

In [10]:
for tr in results.test_results:
    print("=" * 90)
    print("INPUT:", tr.input)
    print("ACTUAL OUTPUT:", tr.actual_output)
    for md_ in tr.metrics_data:
        verdict = "PASS" if md_.success else "FAIL"
        score = f"{md_.score:.2f}" if md_.score is not None else "ERRORED"
        print(f"  [{verdict}] {md_.name}: {score}")
        print(f"      reason: {md_.reason or md_.error}")

INPUT: Run this and tell me the output: print(2**10)
ACTUAL OUTPUT: The output is: 1024
  [PASS] Tool Correctness: 1.00
      reason: [
	 Tool Calling Reason: All expected tools ['run_code'] were called (order not considered).
	 Tool Selection Reason: No available tools were provided to assess tool selection criteria
]

  [FAIL] Argument Correctness: ERRORED
      reason: Missing variable during template render: 'stringified_tools_called' is undefined
  [FAIL] Task Completion [GEval]: ERRORED
      reason: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 274.544244ms.', 'status': 'RESOURC

## Bonus: tracing with `@observe` (optional)

Wrap each tool with `@observe(type="tool")` and, if you set a `CONFIDENT_API_KEY`, DeepEval
uploads the full agent → tool trace to Confident AI while the agent runs — handy for seeing
*exactly* which step went wrong, not just a final score.

```python
from deepeval.tracing import observe

@observe(type="tool")
def search_docs(query: str = "") -> str:
    ...
```

We skip wiring that up here to keep this notebook dependency-free; see the CI/CD guide and the
Streamlit demo app for a worked example.

## Takeaway

`ToolCorrectnessMetric` alone would tell you the third case is "correct" once both tools appear —
but `ArgumentCorrectnessMetric` catches an agent that calls `search_docs(query="")` with an empty
query, and the `GEval` catches one that searches the docs but never actually escalates. Stacking a
deterministic metric with judge-based metrics is how you catch both *wrong actions* and
*incomplete outcomes*.

Next: [`03_chatbot_conversation_evals_with_deepeval.ipynb`](03_chatbot_conversation_evals_with_deepeval.ipynb)
— evaluating a whole conversation, not just one turn.